In [1]:
import pandas as pd
import re
import json
import tqdm

### READ META INFO

In [2]:
# gene info: uniprot_mapping.tsv
# columns: Entry (UniProt) | Gene Names | Unnamed: 2..7 (alternate names / PP_ locus tags)
gene_info = pd.read_csv('../../../raw/PPU/uniprot_mapping.tsv', sep='\t')
name2uniid = {}  # NOTE: use lower case
id2uniid   = {}  # PP_id -> uni_id (for GEM gene rules, PPI, KO)
all_gene_uniid = set(gene_info['Entry'].unique())
for _, row in gene_info.iterrows():
    uni = row['Entry']
    # primary gene name
    if pd.notna(row['Gene Names']):
        name2uniid[str(row['Gene Names']).lower()] = uni
    # scan all other columns for PP_ locus tags or alternate gene names
    for val in row[1:]:
        if pd.notna(val):
            s = str(val)
            if re.match(r'^PP_', s):
                id2uniid[s] = uni
            else:
                name2uniid[s.lower()] = uni

In [3]:
# reaction info
rxn_info_df = pd.read_csv('../../../raw/shared/reaction_info.txt', sep='\t')
rxnbigg = {}
for i in range(rxn_info_df.shape[0]):
    new_bigg = rxn_info_df.iloc[i]['bigg_id']
    old_bigg = rxn_info_df.iloc[i]['old_bigg_ids'].split('; ')
    old_bigg.append(new_bigg)
    for x in old_bigg:
        rxnbigg[x] = new_bigg

In [4]:
# metabolite info
met_info_df = pd.read_csv('../../../raw/shared/metabolite_info.txt', sep='\t')
metabigg = {}
for i in range(met_info_df.shape[0]):
    new_bigg = met_info_df.iloc[i]['bigg_id']
    try:
        old_bigg = met_info_df.iloc[i]['old_bigg_ids'].split('; ')
        old_bigg.append(new_bigg)
    except:
        old_bigg = [new_bigg]
    for x in old_bigg:
        metabigg[x] = new_bigg

inchikey2id = {}
for i in range(met_info_df.shape[0]):
    met_bigg_id = met_info_df.iloc[i]['bigg_id']
    if met_bigg_id.endswith('_c'):
        inchikey = met_info_df.iloc[i]['inchikey']
        if pd.isna(inchikey):
            continue
        inchikey2id['-'.join(inchikey.split('-')[:-1])] = met_bigg_id

In [5]:
# 流通代谢物
common_meta_df = pd.read_csv('../../../raw/shared/common_metabolite.txt', sep='\t')
common_meta = list(common_meta_df['met_id_in_1515'].unique())
common_meta = [metabigg[x] if x in metabigg.keys() else x for x in common_meta]
common_meta_base = {m.rsplit('_', 1)[0] for m in common_meta}

In [6]:
# GEM model (iJN1463)
with open('../../../raw/PPU/iJN1463.json', 'r', encoding='utf-8') as file:
    gem = json.load(file)

# Supplement id2uniid with UniProt annotations embedded in the GEM itself
for g in gem['genes']:
    if g['id'] not in id2uniid and 'annotation' in g and 'uniprot' in g['annotation']:
        prot_uni = g['annotation']['uniprot']
        id2uniid[g['id']] = prot_uni
        all_gene_uniid.add(prot_uni)

### GENERATE TRIPLES

In [7]:
# reaction triples: Reaction --Produces/Consumes--> Metabolite (skip common metabolites, skip BIOMASS)
# reversible reactions (lower_bound < 0) are expanded with a _-1 reverse direction
triples = []

for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    for k, v in x['metabolites'].items():
        k = metabigg.get(k, k)
        if k in common_meta or k.rsplit('_', 1)[0] in common_meta_base:
            continue
        if v > 0:
            triples.append(('Reaction:' + rxn_id, 'Produces', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Consumes', 'Metabolite:' + k))
        elif v < 0:
            triples.append(('Reaction:' + rxn_id, 'Consumes', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Produces', 'Metabolite:' + k))

100%|██████████| 2927/2927 [00:00<00:00, 67140.98it/s]


In [8]:
# catalyze triples: Protein --Catalyzes--> Reaction
# 'and' rules are expanded to individual proteins
# reversible reactions also get a _-1 entry
# NOTE: PPU gene IDs in iJN1463 use PP_ locus tag format (e.g. PP_1602)
for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    if x['gene_reaction_rule'] == '':
        continue
    mets_mapped = [metabigg.get(k, k) for k in x['metabolites']]
    if all(m in common_meta or m.rsplit('_', 1)[0] in common_meta_base for m in mets_mapped):
        continue

    for rule_token in x['gene_reaction_rule'].split(' or '):
        if 'and' in rule_token:
            locus_tags = re.findall(r'PP_\w+', rule_token)
            proteins = [id2uniid[lt] for lt in locus_tags if lt in id2uniid]
        else:
            locus_tag = rule_token.strip().strip('()')
            try:
                proteins = [id2uniid[locus_tag]]
            except:
                continue
        for prot in proteins:
            triples.append(('PPU_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id))
            if reversible:
                triples.append(('PPU_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id + '_-1'))

100%|██████████| 2927/2927 [00:00<00:00, 47889.37it/s]


In [9]:
# ppi triples: Protein --PPI--> Protein  (STRING DB, combined_score >= 800)
ppi_raw_df = pd.read_csv('../../../raw/PPU/160488.protein.links.full.v12.0.txt', sep=' ')
ppi_df = ppi_raw_df[ppi_raw_df['combined_score'] >= 800]
for i in tqdm.tqdm(range(ppi_df.shape[0])):
    try:
        src_uni = id2uniid[ppi_df.iloc[i]['protein1'].split('.')[1]]
        dst_uni = id2uniid[ppi_df.iloc[i]['protein2'].split('.')[1]]
    except KeyError:
        continue
    triples.append(('PPU_Protein:' + src_uni, 'PPI', 'PPU_Protein:' + dst_uni))

100%|██████████| 40076/40076 [00:04<00:00, 8611.04it/s] 


In [10]:
# protein GO triples
# NOTE: PPU protein_go_annotation.tsv has an extra 'SYMBOL' column (same format as BSU)
prot_go_df = pd.read_csv('../../../raw/PPU/protein_go_annotation.tsv', sep='\t')
prot_go_df = prot_go_df[prot_go_df['GENE PRODUCT ID'].isin(all_gene_uniid)]
prot_go_df = prot_go_df[prot_go_df['QUALIFIER'].isin(
    ['enables', 'involved_in', 'part_of', 'located_in']
)]
for i in range(prot_go_df.shape[0]):
    entity_id = prot_go_df.iloc[i]['GENE PRODUCT ID']
    qualifier  = prot_go_df.iloc[i]['QUALIFIER'].capitalize()
    go_term    = prot_go_df.iloc[i]['GO TERM']
    triples.append(('PPU_Protein:' + entity_id, qualifier, go_term))

In [11]:
# protein KO triples (KEGG)
# NOTE: PPU entries use PP_ locus tag format (e.g. ppu:PP_5355)
ko_df = pd.read_csv('../../../raw/shared/genes_ko.list', sep='\t', names=['gene', 'ko'])
ko_df = ko_df[ko_df['gene'].str.startswith('ppu:')]
for i in range(ko_df.shape[0]):
    try:
        prot_uni = id2uniid[ko_df.iloc[i]['gene'].split(':')[1]]
    except:
        continue
    ko_id = 'KO:' + ko_df.iloc[i]['ko'].split(':')[1]
    triples.append(('PPU_Protein:' + prot_uni, 'Has_KO_Item', ko_id))

In [12]:
# common_triples: Metabolite --Belongs_to--> Pathway, etc.
# Filter: drop triples where any Metabolite: node is not in the original KG
kg_mets = set()
for h, r, t in triples:
    if h.startswith('Metabolite:'):
        kg_mets.add(h)
    if t.startswith('Metabolite:'):
        kg_mets.add(t)

common_triples_df = pd.read_csv('../../../raw/shared/common_triples.txt', sep='\t', names=['from', 'rel', 'to'])

def _met_in_kg(row):
    for val in (row['from'], row['to']):
        if val.startswith('Metabolite:') and val not in kg_mets:
            return False
    return True

common_triples_filtered = common_triples_df[common_triples_df.apply(_met_in_kg, axis=1)]
print(f"common_triples: {len(common_triples_df)} total -> {len(common_triples_filtered)} after filtering")

for _, row in common_triples_filtered.iterrows():
    triples.append((row['from'], row['rel'], row['to']))

common_triples: 2919 total -> 2076 after filtering


### OUTPUT kg.txt

In [13]:
kg = pd.DataFrame(triples, columns=['from', 'rel', 'to'])
kg = kg.drop_duplicates()
print(kg.shape)
kg.head()

(77166, 3)


,from,rel,to
0,Reaction:3HAD160_1,Consumes,Metabolite:3hpalmACP_c
1,Reaction:3HAD160_1,Produces,Metabolite:tpalm2eACP_c
2,Reaction:13DAMPPabcpp,Produces,Metabolite:13dampp_c
3,Reaction:13DAMPPabcpp,Consumes,Metabolite:13dampp_p
4,Reaction:13DAMPPtex,Consumes,Metabolite:13dampp_e


In [14]:
kg.to_csv('../../../processed/kg/PPU/kg_reproduced.txt', sep='\t', index=False, header=False)